# Camada Silver

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [0]:
catalog = "rocket_lab"

bronze_schema_name = "vcredit_bronze"
bronze_schema = f"{catalog}.{bronze_schema_name}"

silver_schema_name = "vcredit_silver"
silver_schema = f"{catalog}.{silver_schema_name}"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {silver_schema}")

### 1) Tabela Base de Chamados

Atenção a duas coisas: 

* As colunas hora_abertura_chamado e hora_finalização_atendimento deverão ser preenchidas quando for criar as dimensões na camada gold com o cruzamento da tabela Base de Chamados com Base de Chamados Hora;

* A coluna flag_atendente_placeholder foi criada para sinalizar os chamados cujo id_atendente assume os valores 0, 99 ou 999, que não correspondem a nenhum atendente do cadastro (IDs de 1 a 25) e provavelmente representam códigos de “atendente não identificado” ou erros de registro. Em vez de apagar ou alterar esses valores na camada Silver, optamos por preservá-los e apenas marcá-los, pois a confirmação de que são placeholders depende do cruzamento com a tabela de atendentes, realizado na camada Gold. Essa abordagem mantém a evidência da inconsistência, permite quantificar seu impacto nos indicadores e garante a rastreabilidade do dado original.

In [0]:
# Importação da tabela chamado da camada bronze
bronze_chamado = spark.table(f"{bronze_schema}.chamados")

# Transformação e padronização da coluna id_camado
silver_chamado = (
    bronze_chamado
    .withColumn("id_chamado", F.expr("try_cast(trim(id_chamado) as bigint)"))
    .dropDuplicates(["id_chamado"])
)

# Transformação e padronização da coluna id_cliente
silver_chamado = (
    silver_chamado
    .withColumn("id_cliente", F.trim(F.col("id_cliente")))
)

COM_ACENTO = "áàâãäéèêëíìîïóòôõöúùûüçñÁÀÂÃÄÉÈÊËÍÌÎÏÓÒÔÕÖÚÙÛÜÇÑ"
SEM_ACENTO = "aaaaaeeeeiiiiooooouuuucnAAAAAEEEEIIIIOOOOOUUUUCN"

# Transformação e padronização da coluna motivo
silver_chamado = (
    silver_chamado.
    withColumn("motivo", F.lower(
        F.regexp_replace(
            F.trim(F.translate(F.regexp_replace(F.col("motivo"), "[\\u0300-\\u036f]", ""), COM_ACENTO, SEM_ACENTO)), r"\s+", " "
        )))
)

# Transformação e padronização da coluna canal
silver_chamado = (
    silver_chamado
    .withColumn("canal", F.when(F.col("canal").isin(["Ura", "U.R.A", "URA"]), "ura")
                          .when(F.col("canal").isin(["CHATBOT", "ChatBot", "Chat bot", "Chatbot", "chat bot", "Chat Bot"]), "chatbot")
                          .when(F.col("canal").isin(["ATENDIMENTO INICIAL", "Atend. Inicial", "Atendimento_Inicial", "Atendimento Inicial"]), "atendimento inicial")
                          .when(F.col("canal").isin(["ATENDIMENTO ESPECIALIZADO", "Atend. Especializado", "Atendimento_Especializado", "Atendimento Especializado"]), "atendimento especializado")
                          .otherwise(F.col("canal")))
)

# Transformação e padronização da coluna resolvido
silver_chamado = (
    silver_chamado
    .withColumn("resolvido", F.when(F.col("resolvido").isin(["Sim", "S", "1"]), "sim")
                          .when(F.col("resolvido").isin(["Não", "N", "0", "Nao resolvido"]), "nao")
                          .otherwise(F.col("resolvido")))
)

# Transformação e padronização da coluna hora_inicio_atendimento
silver_chamado = (
    silver_chamado
    .withColumn("hora_inicio_atendimento", F.lower(F.trim(F.col("hora_inicio_atendimento"))))
)

# Transformação e padronização da coluna tempo_espera
silver_chamado = (
    silver_chamado
    .withColumn("tempo_espera", F.expr("try_cast(nullif(upper(trim(tempo_espera)), 'NULL') as int)"))
    .withColumn("tempo_espera", F.when(F.col("tempo_espera") <= 0, None).otherwise(F.col("tempo_espera")))
)

# Transformação e padronização da coluna tempo_atendimento
silver_chamado = (
    silver_chamado
    .withColumn("tempo_atendimento", F.expr("try_cast(nullif(upper(trim(tempo_atendimento)), 'NULL') as int)"))
    .withColumn("tempo_atendimento", F.when(F.col("tempo_atendimento") <= 0, None).otherwise(F.col("tempo_atendimento")))
)

# Transformação e padronização da coluna id_atendente
# Adição de uma nova coluna flag_atendente_placeholder para verificar, na camada gold, os IDs de atendentes com valores 0, 99 e 999
silver_chamado = (
    silver_chamado
    .withColumn("id_atendente", F.expr("try_cast(trim(id_atendente) as int)"))
    .withColumn("flag_atendente_placeholder", F.col("id_atendente").isin([0, 99, 999]))
)

 #idempotente
(
    silver_chamado.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{silver_schema}.chamados")
)

display(silver_chamado.orderBy("id_chamado").limit(10))


### 2) Base de Pesquisas de Satisfação

A coluna qtd_chamados_pesquisa foi criada para registrar quantas pesquisas de satisfação existem para um mesmo id_chamado, já que identificamos algumas chamados com duas respostas, todas com a mesma nota, mas com id_pesquisa diferentes. Em vez de remover essas repetições na camada Silver, optamos por contá-las e preservá-las, pois isso mantém a evidência da inconsistência, permite quantificar seu impacto nos indicadores de satisfação (que seriam distorcidos se esses chamados pesassem em dobro na média) e deixa a decisão de como deduplicar para a camada Gold, onde o cruzamento com a base de chamados é feito.

In [0]:
# Importação da tabela pesquisa da camada bronze
bronze_pesquisa = spark.table(f"{bronze_schema}.pesquisa")

# Particionamento por id_chamado
w = Window.partitionBy("id_chamado")

# Tratamento e padronização dos dados da tabela pesquisa
silver_pesquisa = (
    bronze_pesquisa
    .withColumn("id_pesquisa", F.expr("try_cast(trim(id_pesquisa) as bigint)"))
    .withColumn("id_chamado", F.expr("try_cast(trim(id_chamado) as bigint)"))
    .withColumn("qtd_chamados_pesquisa", F.count("*").over(w))
    .withColumn("nota_atendimento", F.expr("try_cast(nullif(upper(trim(nota_atendimento)), 'NULL') as int)"))
    .withColumn("nota_atendimento", F.when((F.col("nota_atendimento") <= 0) | (F.col("nota_atendimento") > 5), None).otherwise(F.col("nota_atendimento")))
)

 #idempotente
(
    silver_pesquisa.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{silver_schema}.pesquisa")
)

display(silver_pesquisa.orderBy("id_pesquisa").limit(10))

### 3) Base de Custos

In [0]:
# Importação da tabela custos da camada bronze
bronze_custos = spark.table(f"{bronze_schema}.custos")

# Tratamento e padronização dos dados da tabela custos
silver_custos = (
    bronze_custos
    .withColumn("id_custo", F.expr("try_cast(trim(id_custo) as bigint)"))
    .withColumn("id_chamado", F.expr("try_cast(trim(id_chamado) as bigint)"))
    .dropDuplicates(["id_custo"])
    .withColumn("custo", F.regexp_replace(F.trim("custo"), r"[^0-9,.\-]", "")) 
    .withColumn("custo", F.regexp_replace("custo", ",", "."))                  
    .withColumn("custo", F.expr("try_cast(custo as decimal(18,2))"))
)

 #idempotente
(
    silver_custos.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{silver_schema}.custos")
)

display(silver_custos.orderBy("id_custo").limit(10))

### 4) Base de Clientes

Uma coluna de faixa etária foi criada para preservar as informações pessoais de idade do cliente quando formos trabalhar com a camada gold

In [0]:
# Importação da tabela clientes da camada bronze
bronze_clientes = spark.table(f"{bronze_schema}.clientes")

silver_clientes = (
    bronze_clientes
    .withColumn("idade", F.expr("try_cast(trim(idade) as int)"))
    .where((F.col("idade").isNotNull()) & (F.col("idade").between(18, 110)))
    .withColumn("id_cliente", F.trim("id_cliente"))
    .dropDuplicates(["id_cliente"])
    .withColumn("nome", F.initcap(F.regexp_replace(F.trim("nome"), r"\s+", " ")))

    # remove espaços internos ("@hotmail . com") e valida o formato
    .withColumn("email", F.lower(F.regexp_replace("email", r"\s+", "")))
    .withColumn("email", F.when(F.col("email").rlike(r"^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$"), F.col("email")).otherwise(None))
    .withColumn("regiao", F.lower(F.trim(F.col("regiao"))))
    
    # cria a coluna para a faixa etária 
    .withColumn("faixa_etaria",
        F.when(F.col("idade") < 25, "18-24").when(F.col("idade") < 35, "25-34")
         .when(F.col("idade") < 45, "35-44").when(F.col("idade") < 55, "45-54")
         .when(F.col("idade") < 65, "55-64").otherwise("65+"))
    .withColumn("data_cadastro", F.coalesce(
        F.expr("try_to_date(data_cadastro, 'dd/MM/yyyy')"),
        F.expr("try_to_date(data_cadastro, 'yyyy-MM-dd')")))
    .withColumn("dia_vencimento_fatura", F.expr("try_cast(dia_vencimento_fatura as int)"))
    .withColumn("dia_vencimento_fatura", F.when((F.col("dia_vencimento_fatura") <= 0) | (F.col("dia_vencimento_fatura") > 31), None).otherwise(F.col("dia_vencimento_fatura")))
)

 #idempotente
(
    silver_clientes.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{silver_schema}.clientes")
)

display(silver_clientes.orderBy("id_cliente").limit(10))
